# QCC | J5 chemistry loaders + deeper lookup plans

**Our part: the chemistry ANGLE circuits, recompiled with five words per circuit, now on Ahmed Berat Özer's
re-quantized C46 table.** Two changes to the table in Özer's notebook (version 9: Michael Rustad's R = 10, B = 129,
C = 46 factorization with Özer's quantization and λ loop): the lookup circuits that load its words are recompiled with
five words per circuit, and its weights W and H2_I are refit with every word held fixed, so the same circuits stay valid
while λ_eff falls from 47.553 to 47.494 (error_1 19.9990 mHa, error_2 19.9989 mHa, E_gap 3.914908 against the 3.914898
floor).

| Chemistry build | Table | Circuits | Toffolis per circuit per pass | ANGLES T (both passes) | Total T | Q | Ratio |
|---|---|---:|---:|---:|---:|---:|---:|
| Özer's notebook, J = 5 | Özer C46 | 15 | 321 | 9,624 | 17,330 | 431 | 9.4474 |
| this notebook's previous build, J = 5 | C46 + our refit | 15 | 312 | 9,360 | 17,066 | 431 | 9.5258 |
| J = 5 on Özer's weights | Özer C46 | 15 | 312 | 9,360 | 17,066 | 431 | 9.5936 |
| **this version: J = 5 + refit weights (λ_eff 47.494)** | **Özer C46** | **15** | **312** | **9,360** | **17,066** | **431** | **9.6055** |

Five words need 40 word qubits plus the 11-qubit selection register and a monomial cache, which does not fit the
128-qubit limit for ANGLE circuits as one cache. Three things make it fit and keep it cheap:
- **Multi-phase low cache.** The 11 cached low-address monomials are split into three groups and cached one group at a
  time; every paired product is planned inside the address bits plus one group, so the low cache peaks at four qubits.
- **Word-top-bit scratch.** The angle words have equal top two bits (a redundant sign bit; true on every row of this
  table too), so one CNOT from bit 6 clears bit 7 of a word register, which then serves as a clean scratch qubit and is
  restored at the end. The official verifier feeds each ANGLE circuit only the table's own words (the previous batch
  forward, its own batch in reverse), independent of the salt, so this holds on every split.
- **Searched relation layout.** A circuit costs 78 high-address cache Toffolis, 11 low-cache Toffolis and one Toffoli per
  paired product; every linear relation among the products saves products. We search how the low monomials are grouped
  and which relation types each group carries (per-group frontiers certified by exact GF(2) matroid parity, combined
  under the address-bit budget, plus relations on the address-bit block), then close the last gaps with a relation-level
  local search on the exact GF(2) deficiency: 79 relations per circuit, 223 products, 312 Toffolis. The product count
  does not depend on the table, so the layout carries over to any table with this shape.

Every circuit is exact (128 qubits, every Toffoli unconditional, so the count is the same on every salt) and passed the
official verifier on two salts before assembly.

| Task | Circuit | Q | Mean Toffolis | Ratio (default empty salt) |
|---|---|---:|---:|---:|
| invsqrt | mrmeows' 63-qubit build (dataset version 3) | 63 | 1,348.5 | 20.057 |
| DQI | SomeJoint's 883-qubit decoder with GPT-6 Astra Pro's 658-Toffoli jet | 883 | 467,789.6 | 55.991 |
| dfthcsos | Özer's C46 table with our refit weights, our J = 5 circuits | 431 | 17,066 | 9.606 |

**Leaderboard expectation.** invsqrt and DQI are byte-identical to the files in SomeJoint's 21.98 notebook (leaderboard 22.00949), whose chemistry is Özer's version-9 circuits (321 Toffolis per circuit) with busyaprime's refit weights (λ_eff 47.537, ratio 9.4507); the chemistry cost does not depend on the salt, so the expected score is 22.00949 × (9.6055 / 9.4507)^(1/3) ≈ 22.13.

## Local check before publishing

Kaggle's own metric code (the `google-quantum-ai-metric` notebook, unchanged) run end to end on these exact files with two local salts gave **22.0953** and **22.1291** (split 0 below). The leaderboard uses secret salts; only the DQI cost moves with the salt, through its repetition factor.

| Task | Cost | Ratio (reference / cost) |
|---|---:|---:|
| invsqrt | 103546 | 20.0565 |
| dqi | 4.36484e+11 | 55.9916 |
| dfthcsos | 3.4934e+08 | 9.6055 |

## Re-verify every component here

Each file is copied from the release dataset and matched to its SHA-256; then the three official verifiers from the competition data run on it, unmodified, at their default shot counts.

In [ ]:
import glob, hashlib, json, lzma, math, os, shutil, sys, time, zipfile, pathlib

def find_comp():
    for p in [os.environ.get('QCC_COMP_DIR', ''), '/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation',
              '/kaggle/input/google-verifiable-quantum-circuit-compilation']:
        if p and os.path.isdir(os.path.join(p, 'verifiers')):
            return p
    raise SystemExit('competition data not found')

def find_release():
    if os.environ.get('QCC_DS_DIR'):
        return os.environ['QCC_DS_DIR']
    hits = glob.glob('/kaggle/input/**/QCC_RELEASE_MANIFEST.json', recursive=True)
    if not hits:
        raise SystemExit('release dataset not found')
    return os.path.dirname(hits[0])

COMP, DS = find_comp(), find_release()
WORK = pathlib.Path(os.environ.get('QCC_WORK_DIR', '/kaggle/working'))
SUB = WORK / 'submission'
sys.path[:0] = [os.path.join(COMP, 'kickmix'), os.path.join(COMP, 'verifiers')]
import kickmix as km
from invsqrt import verify_submission_invsqrt as vinv
from dqi import verify_submission_dqi as vdqi
from dfthcsos import verify as vdft
MANIFEST = json.load(open(os.path.join(DS, 'QCC_RELEASE_MANIFEST.json')))
EXPECTED_PORTFOLIO = 'v12_sj883L5'
assert MANIFEST['portfolio'] == EXPECTED_PORTFOLIO, ('stale dataset version', MANIFEST['portfolio'], EXPECTED_PORTFOLIO)
SALT = MANIFEST['salt']
print('competition data:', COMP, '| release dataset:', DS, '| salt:', SALT)

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 22), b''):
            h.update(chunk)
    return h.hexdigest()

if SUB.exists():
    shutil.rmtree(SUB)
for item in MANIFEST['files']:
    src, dst = os.path.join(DS, item['stored_as']), SUB / item['file']
    dst.parent.mkdir(parents=True, exist_ok=True)
    if item['stored_as'].endswith('.xz'):
        with lzma.open(src) as fi, open(dst, 'wb') as fo:
            shutil.copyfileobj(fi, fo, 1 << 22)
    else:
        shutil.copyfile(src, dst)
    got = sha256(dst)
    assert got == item['sha256'], (item['file'], got)
print(len(MANIFEST['files']), 'files copied and matched to the manifest by SHA-256')

In [ ]:
REF = MANIFEST['reference_costs']
results = {}
t0 = time.time()
circ = km.Circuit.from_file(str(SUB / 'invsqrt.kmb'), format='kmb')
results['invsqrt'] = vinv.verify(circ, vinv.ProblemSpecInvSqrt.for_n_p(10), salt=SALT)
print('invsqrt done', round(time.time() - t0), 's'); t0 = time.time()
circ = km.Circuit.from_file(str(SUB / 'dqi.kmb'), format='kmb')
results['dqi'] = vdqi.verify(circ, vdqi.ProblemSpecDQI.benchmark_instance(), salt=SALT)
del circ
print('dqi done', round(time.time() - t0), 's'); t0 = time.time()
results['dfthcsos'] = vdft.verify(str(SUB / 'dfthcsos'), salt=SALT)
print('dfthcsos done', round(time.time() - t0), 's')
rows, ratios = [], []
for task in ('invsqrt', 'dqi', 'dfthcsos'):
    r = results[task]
    assert r['status'] == 'PASS', (task, r)
    ratio = REF[task] / float(r['score'])
    ratios.append(ratio)
    if task == 'invsqrt':
        extra = f"{r['num_qubits']} qubits, {r['toffoli_equivalent']:.1f} Toffoli-equivalents"
    elif task == 'dqi':
        extra = (f"{r['num_qubits']} qubits, {r['toffoli_count']:.0f} mean Toffolis, "
                 f"{r.get('value_errors', 0)} value / {r.get('phase_errors', 0)} phase errors")
    else:
        extra = f"lambda_eff {r['lambda_eff']:.3f}, {r['toffoli_count']:.0f} Toffolis, {r['num_qubits']} qubits"
    rows.append((task, float(r['score']), ratio, extra))
    print(f"{task:9s} PASS cost {float(r['score']):.6g}  ratio {ratio:8.4f}  ({extra})")
gm = math.prod(ratios) ** (1 / 3)
print(f'geometric mean of reference / cost: {gm:.4f}')

## Write the submission

In [ ]:
zip_path = WORK / 'submission.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for item in MANIFEST['files']:
        z.write(SUB / item['file'], item['file'])
shutil.rmtree(SUB)
print('wrote', zip_path, round(zip_path.stat().st_size / 2**20, 1), 'MiB')

# Sources and credits

- **invsqrt** — mrmeows' 63-qubit build from dataset version 3 (1,348.5 mean Toffolis, 20.06x;
  https://www.kaggle.com/datasets/mrmeows/qcc-1375-open-circuits, Apache-2.0, notebook
  https://www.kaggle.com/code/mrmeows/21-0-circuits-let-s-all-reach-the-plateau), byte for byte (sha256 0766a111…);
  lineage Akmal Xodarev (busyaprime), Phuc Thai (DePie), Dmitrii Gluzdov, marsyas.
- **dqi** — SomeJoint's (Agniv Banerjee's team) 883-qubit decoder with GPT-6 Astra Pro's 658-Toffoli jet
  (https://www.kaggle.com/code/somejoint/quantum-circuits-21-98-local-gm-883-qubit-dqi), byte for byte from that
  notebook's output (sha256 ef334fee…). Lineage as its authors name it: SomeJoint's team (the confluent/fused decoder,
  rebuilt from a pseudocode description of Alex Chilton's 883-qubit Schur decoder, which is based on jday96314's
  QuantumAI decoder, https://github.com/jday96314/QuantumAI); Ahmed Berat Özer (narrowing ideas); mrmeows and
  manwithacat (mask projection); Dmitrii Gluzdov, ViberTeam, Phuc Thai (DePie).
- **dfthcsos** — the table is the one in Özer's notebook output (quantum-ai-df-thc-group-compression, version 9): Michael Rustad's R = 10,
  B = 129, C = 46 factorization with Ahmed Berat Özer's quantization and λ loop (λ_eff 47.553), building on mrmeows'
  compiler foundations and one-body tool, kozykappa's joint refine, marculera's four-address loaders and Vladislav
  Lavrentev / lightningv08's five-word schedule. Ours on top: the 15 ANGLE circuits recompile exactly those words with
  five words per circuit at 312 Toffolis per circuit (his notebook's own J = 5 build uses 321), and W and H2_I are refit
  with every word fixed (λ_eff 47.553 → 47.494), building on Michael Rustad's Apache-2.0 metric and gauge code (see below).
- **Ours** — the J = 5 ANGLE compiler (multi-phase low cache, word-top-bit scratch, and an exact layout search over
  low-monomial groupings and relation types, then a relation-level search: 79 relations, 312 Toffolis per circuit per pass), the assembly, and the
  verification notebook.
- **Organizers** — task definitions, verifiers and kickmix (Google Quantum AI / Qualtran, competition data).